### Importing Libraries

In [ ]:
import math
import torch
import torch.nn as nn
from google.colab import drive
from torch.nn import functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR

# Mount Google Drive to save/load models
drive.mount('/content/drive/')

### Loading the dataset

In [ ]:
# Load Shakespeare text from Google Drive
with open('/content/drive/MyDrive/data/tinyshakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Total characters in dataset: {len(text):,}")
print("\nFirst 500 characters:\n")
print(text[:500])

#### Explore the dataset

In [ ]:
# understand what model will learn
chars = sorted(list(set(text)))
vocab_size_char = len(chars)

print(f"Unique characters: {vocab_size_char}")
print(f"Characters: {''.join(chars)}")
print(f"\nTotal characters: {len(text):,}")
print(f"Estimated tokens (BERT): ~{len(text)//4:,}")  # Rough estimate

## Choose Model Type 

In [ ]:
# Change this to "bert" or "character" and re-run notebook

#MODEL_TYPE = "bert"  # Options: "bert" or "character"
MODEL_TYPE = "character"
print(f"Training model type: {MODEL_TYPE}")

## Tokenization Layer

In [ ]:
from transformers import AutoTokenizer

if MODEL_TYPE == "character":
    # Character-level: each character = one token 
    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    stoi = {ch: i for i, ch in enumerate(chars)}  # char → index
    itos = {i: ch for i, ch in enumerate(chars)}  # index → char
    tokens = [stoi[ch] for ch in text]
    print(f"Character model - vocab_size: {vocab_size}")
    print(f"Sample mapping: 'a' → {stoi['a']}, 'z' → {stoi['z']}")
    
else:  # BERT
    # BERT: subword tokens (better for real text quality)
    tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
    vocab_size = tokenizer.vocab_size  # ← FIXED: ~30522, NOT len(tokens)!
    tokens = tokenizer.encode(text)
    print(f"BERT model - vocab_size: {vocab_size}")
    print(f"Sample tokens (first 10): {tokens[:10]}")

# Convert to torch tensor
data = torch.tensor(tokens, dtype=torch.long)
print(f"Total tokens: {len(data):,}")

#### Split up the data into train and validation sets

In [ ]:
# Split data: 90% train, 10% validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens: {len(val_data):,}")

# Verify shift - target is input shifted by 1 position
print(f"\nFirst 10 train tokens: {train_data[:10]}")
print(f"Target tokens (shifted): {train_data[1:11]}")

### Define Hyperparameters

In [ ]:
batch_size = 16
block_size = 128
max_iters = 10000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128
n_head = 8
n_layer = 8
dropout = 0.1  # (prevents overfitting)

#### generate batch of data X and Y

In [ ]:
def get_batch(split):
    """Randomly sample batches of input (x) and target (y) sequences"""
    data = train_data if split == 'train' else val_data
    
    # Pick random starting positions
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # x = input sequence, y = same sequence shifted by 1 (target)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
x, y = get_batch('train')

print("Input context window (x):\n")

if MODEL_TYPE == "character":
    # Decode character-level
    decoded = ''.join([itos[int(idx)] for idx in x[1].tolist()])
    print(decoded)
    print("\n\nTarget output (y - shifted by 1):\n")
    decoded_y = ''.join([itos[int(idx)] for idx in y[1].tolist()])
    print(decoded_y)
else:  # BERT
    print(tokenizer.decode(x[1].tolist()))
    print("\n\nTarget output (y - shifted by 1):\n")
    print(tokenizer.decode(y[1].tolist()))

### Implementing a layer to calculate loss

In [ ]:
@torch.no_grad()  # Disables gradient computation (faster, less memory)
def estimate_loss():
    """Evaluate model on train and validation sets without updating weights"""
    out = {}
    model.eval()   # Set to eval mode (disables dropout)
    
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()  # Average loss over eval_iters batches
    
    model.train()  # Back to training mode (enables dropout)
    return out

### Implementing a single attention head layer

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        attention_score = q @ k.transpose(-2, -1) * C**-0.5
        attention_score = attention_score.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        attention_score = F.softmax(attention_score, dim=-1)
        attention_score = self.dropout(attention_score)
        v = self.value(x)
        return attention_score @ v

### Implement the multi Head Attention Layer

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # projects the result of each head into a linear layer
        self.projection = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
      # concatnate the result of each attention head
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # the result of each head is mixed
        out = self.dropout(self.projection(out))
        return out

### FeedForward Network

In [ ]:
class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # Expand 128 → 512
            nn.ReLU(),                       # Non-linearity
            nn.Linear(4 * n_embd, n_embd),  # Contract 512 → 128
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## Transformer Layer

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.mHA = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)  # Renamed for clarity
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.mHA(self.ln1(x))  # Attention + residual
        x = x + self.ffwd(self.ln2(x)) # FFN + residual
        return x

### Decode Only Tokenizer

In [ ]:
class DecodOnlyTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, token, targets=None):
        B, T = token.shape
        token_emb = self.token_embedding(token)
        position_emb = self.position_embedding(torch.arange(T, device=device))
        x = token_emb + position_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            logits = logits.view(B*T, -1)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, token, max_new_tokens):
        for _ in range(max_new_tokens):
            token_cond = token[:, -block_size:]  # Crop to context window
            logits, _ = self(token_cond)
            logits = logits[:, -1, :]  # Last token only
            probs = F.softmax(logits, dim=-1)
            token_next = torch.multinomial(probs, num_samples=1)
            token = torch.cat((token, token_next), dim=1)
        return token

## Train the model

In [ ]:
model = DecodOnlyTransformer()
m = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = CosineAnnealingLR(optimizer, T_max=max_iters)

best_val_loss = float('inf')

for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            # Save with model type in filename
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_val_loss,
                'model_type': MODEL_TYPE,
                'vocab_size': vocab_size,
            }, f'/content/drive/MyDrive/data/decoder_model_{MODEL_TYPE}.pth')
            print(f"Saved {MODEL_TYPE} model at step {iter} | val loss: {best_val_loss:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    scheduler.step()

### Save the model

In [ ]:
# Save final model with type in filename
PATH = f'/content/drive/MyDrive/data/decoder_model_{MODEL_TYPE}.pth'

checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
    'model_type': MODEL_TYPE,
    'vocab_size': vocab_size,
}

torch.save(checkpoint, PATH)
print(f"Model saved to: {PATH}")

### Evaluate the model

In [ ]:
def evaluate_model(model_type="bert"):
    PATH = f'/content/drive/MyDrive/data/decoder_model_{model_type}.pth'
    checkpoint = torch.load(PATH, map_location=device)
    
    model = DecodOnlyTransformer()
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for _ in range(100):
            x, y = get_batch('val')
            logits, loss = model(x, y)
            if loss is not None:
                total_loss += loss.item() * x.numel()
                total_tokens += x.numel()

    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    perplexity = math.exp(avg_loss)

    print(f"Model: {model_type}")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Perplexity: {perplexity:.4f} (lower = better)")

    return {'perplexity': perplexity, 'loss': avg_loss}

In [ ]:
evaluate_model()